In [84]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import math
import os
import os
from ultralytics import YOLO
import easyocr
import sys
import os
import re
import torch
os.chdir(r"C:\Users\andre\Desktop\ALL\Master\AIVA_2026-MUVA")
print(torch.cuda.is_available())

True


### Modelo ya entrenado

In [85]:
model = YOLO("system/data/models/best.pt")
reader = easyocr.Reader(['en'], gpu=True)

### Video

In [ ]:
results_video = model("system/data/videos/Video_11.mkv", save=True)

### Video frame a frame

In [77]:
def add_unique(plate_list, plate_text):
    if plate_text and plate_text not in plate_list:

        prefix6 = plate_text[:6]   # "1234 A"
        first4 = plate_text[:4]    # "1234"
        last5 = plate_text[-5:]    # "4 ABC"

        exists = any(
            p[:6] == prefix6 or
            p[:4] == first4 or
            p[-5:] == last5
            for p in plate_list
        )

        if not exists:
            plate_list.append(plate_text)


In [86]:
cap = cv2.VideoCapture("system/data/videos/Video_10.mkv")
#cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = 0
padding_box = 0
plates_verificadas = []
consonantes = "BCDFGHJKLMNPQRSTVWXYZ"
vowels = "AEIOU"

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Solo procesar cada 2 frames
    if frame_count % 2 == 0:
        #print("frame count: ", frame_count)
        results = model(frame, verbose=False)
        result_frame = results[0]
        # Bucle para recortar
        for box, conf in zip(result_frame.boxes.xyxy, result_frame.boxes.conf):
            if conf < 0.50:
                continue
            # Recortar el box
            x1, y1, x2, y2 = map(int, box)
            x1 = max(0, x1 - padding_box)
            y1 = max(0, y1 - padding_box)
            x2 = min(frame.shape[1], x2 + padding_box)
            y2 = min(frame.shape[0], y2 + padding_box)
            plate_crop = frame[y1:y2, x1:x2]

            # Preprocesado para OCR
            gray = cv2.cvtColor(plate_crop, cv2.COLOR_BGR2GRAY)
            gray = cv2.resize(gray, None, fx=4, fy=4, interpolation=cv2.INTER_CUBIC)
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
            gray = clahe.apply(gray)
            _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

            # Aplicar OCR
            result_text = reader.readtext(thresh, detail=1)
            texts = [f"{text}({conf:.2f})" for bbox, text, conf in result_text]


            print(f"Texto resultado OCR: {texts}")

            # Comprobacion de formato matriula
            if len(result_text) == 1:
                text, conf = result_text[0][1], result_text[0][2]
                if conf > 0.60:
                    text = text.strip().upper()[-8:]
                    if (
                        len(text) == 8 and
                        text[:4].isdigit() and
                        text[4] == " " and
                        len(text[5:]) == 3 and
                        all(c in consonantes for c in text[5:])
                    ):
                        print("Valido para add")
                        add_unique(plates_verificadas, text)
            elif len(result_text) > 2:
                # Filtrar por confianza > 0.50
                filtered = [res for res in result_text if res[2] > 0.50]
                # Ordenar por confianza (de mayor a menor)
                filtered_sorted = sorted(filtered, key=lambda x: x[2], reverse=True)
                # Quedarte con los 2 mejores
                result_text = filtered_sorted[:2]


            if len(result_text) == 2:
                texts_list = [res[1] for res in result_text]
                confs_list = [res[2] for res in result_text]

                max_conf = max(confs_list)
                if max_conf > 0.90:
                    # coger solo el de mayor confianza
                    best_text = max(result_text, key=lambda x: x[2])[1]

                    if (
                        len(best_text) == 8 and
                        best_text[:4].isdigit() and
                        best_text[4] == " " and
                        len(best_text[5:]) == 3 and
                        all(c in consonantes for c in best_text[5:])
                    ):
                        print("Valido para add")
                        add_unique(plates_verificadas, best_text)


                if confs_list[0] > 0.60 and confs_list[1] > 0.60:
                    text1, text2 = texts_list[0].replace(" ", ""), texts_list[1].replace("|", "")
                    text1, text2 = text1.replace("|", "1"), text2.replace("|", "1")

                    is_consonants_1 = text1.isalpha() and all(c not in vowels for c in text1)
                    is_consonants_2 = text2.isalpha() and all(c not in vowels for c in text2)

                    numbers_part, letters_part = None, None
                    if text1.isdigit() and is_consonants_2:
                        numbers_part = text1
                        letters_part = text2
                    elif text2.isdigit() and is_consonants_1:
                        numbers_part = text2
                        letters_part = text1


                    if numbers_part and letters_part:
                        numbers_part_last = numbers_part[-4:]
                        letters_part_first = letters_part[:3]
                        #print("Orden:", numbers_part, letters_part)
                        #print("Corregido:", numbers_part_last, letters_part_first)

                        if len(numbers_part_last) == 4 and len(letters_part_first) == 3:
                            print("Valido para add")
                            add_unique(plates_verificadas, numbers_part_last + " " + letters_part_first.upper())



    frame_count += 1

cap.release()
cv2.destroyAllWindows()

Texto resultado OCR: ['GLN(0.59)', '6638(0.61)']
Texto resultado OCR: ['GLN(0.49)', '4638(0.94)']
Texto resultado OCR: ['014638(0.40)']
Texto resultado OCR: ['GLN(0.98)', '4638(0.93)']
Valido para add
Texto resultado OCR: ['GLN(0.92)', 'L538(0.37)']
Texto resultado OCR: ['GLN(0.95)', 'L638(0.99)']
Texto resultado OCR: ['T@3K(0.06)']
Texto resultado OCR: ['Dw(0.02)']
Texto resultado OCR: ['[w(0.05)']
Texto resultado OCR: ['[22E(0.27)']
Texto resultado OCR: ['4743HHR(0.99)']
Texto resultado OCR: ['HHR(0.99)', '4743(1.00)']
Valido para add
Texto resultado OCR: ['{22(0.43)', '6S(0.18)']
Texto resultado OCR: ['4743 HHR(0.93)']
Valido para add
Texto resultado OCR: ['02220(0.55)', '635(0.32)']
Texto resultado OCR: ['4743 HHR(0.81)']
Valido para add
Texto resultado OCR: ['GRS(0.65)', '02470(0.86)']
Valido para add
Texto resultado OCR: ['43 HHR(0.99)']
Texto resultado OCR: ['GRS(0.88)', '2470(0.98)']
Valido para add
Texto resultado OCR: ['GRS(0.86)', '2470(0.92)']
Valido para add
Texto resultad